In [2]:
from Booleanize import *

In [3]:
import pandas as pd
import glob

csv_files=glob.glob("/Users/devendergupta/Desktop/python_project/tsetlin_machine/local/fractal_maps/*.csv")
filenames = [os.path.splitext(os.path.basename(f))[0].replace('_fd', '') for f in csv_files]


In [4]:
import numpy as np

booleanized_data={}
for i in range(len(csv_files)):
    booleanized_data[filenames[i]]=booleanize_array(np.loadtxt(csv_files[i],delimiter=",").flatten(),5).flatten()

/Users/devendergupta/Desktop/python_project/tsetlin_machine/local/ShellScripted_TA/fractal_env_arm/lib/python3.13/site-packages/sklearn/preprocessing/_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/Users/devendergupta/Desktop/python_project/tsetlin_machine/local/ShellScripted_TA/fractal_env_arm/lib/python3.13/site-packages/sklearn/preprocessing/_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


In [5]:
print(len(booleanized_data))

10015


In [6]:
sample_key = list(booleanized_data.keys())[0]
print(booleanized_data[sample_key].shape)

(19080,)


In [7]:
#splitting the data in x_train and x_test 
from sklearn.model_selection import train_test_split 

x_train,x_test = train_test_split(filenames,test_size=0.2,random_state=21)

In [8]:
cancerous = ['akiec', 'bcc', 'mel']
non_canc = ['bkl', 'df', 'nv', 'vasc']


In [9]:
import pandas as pd

df = pd.read_csv("/Users/devendergupta/Desktop/python_project/tsetlin_machine/local/HAM10000/HAM10000_metadata.csv")

# create a dictionary mapping image_id to dx label
label_dict = dict(zip(df['image_id'], df['dx']))

print(len(label_dict))     # total number of mappings
print(list(label_dict.items())[:5])  # show first 5 mappings

10015
[('ISIC_0027419', 'bkl'), ('ISIC_0025030', 'bkl'), ('ISIC_0026769', 'bkl'), ('ISIC_0025661', 'bkl'), ('ISIC_0031633', 'bkl')]


In [10]:
for _, row in df.iterrows():
    image_id = row['image_id']   # match with your filenames
    diagnosis = row['dx']
    if diagnosis in cancerous:
        label_dict[image_id] = 1
    else:
        label_dict[image_id] = 0

X_train = np.array([booleanized_data[f] for f in x_train], dtype=object) # dtype = np.int32 , this is chatgpt reccomendation, try later
X_test = np.array([booleanized_data[f] for f in x_test], dtype=object)

Y_train = np.array([label_dict[f] for f in x_train])
Y_test = np.array([label_dict[f] for f in x_test])

np.save('X_train.npy', X_train, allow_pickle=True)
np.save('X_test.npy', X_test, allow_pickle=True)
np.save('Y_train.npy', Y_train)
np.save('Y_test.npy', Y_test)


print("✅ Saved: X_train.npy, X_test.npy, Y_train.npy, Y_test.npy, booleanized_data.npy")
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

✅ Saved: X_train.npy, X_test.npy, Y_train.npy, Y_test.npy, booleanized_data.npy
Train size: 8012, Test size: 2003


In [11]:
print(X_train.shape, Y_train.shape)
print(X_test.shape, Y_test.shape)

(8012, 19080) (8012,)
(2003, 19080) (2003,)


In [12]:
from sklearn.utils import resample
import numpy as np

# Separate classes
X_non = X_train[Y_train == 0]
Y_non = Y_train[Y_train == 0]

X_cancer = X_train[Y_train == 1]
Y_cancer = Y_train[Y_train == 1]

print("\nBefore balancing:")
print("Non-cancer:", len(X_non))
print("Cancer:", len(X_cancer))

# Oversample cancer class
X_cancer_up, Y_cancer_up = resample(
    X_cancer,
    Y_cancer,
    replace=True,
    n_samples=len(X_non),
    random_state=42
)

# Combine
X_train_bal = np.concatenate((X_non, X_cancer_up), axis=0)
Y_train_bal = np.concatenate((Y_non, Y_cancer_up), axis=0)

# Shuffle
indices = np.random.permutation(len(Y_train_bal))
X_train_bal = X_train_bal[indices]
Y_train_bal = Y_train_bal[indices]

print("\nAfter balancing:")
print("Non-cancer:", np.sum(Y_train_bal == 0))
print("Cancer:", np.sum(Y_train_bal == 1))

# Save balanced dataset
np.save('X_train_bal.npy', X_train_bal, allow_pickle=True)
np.save('Y_train_bal.npy', Y_train_bal)

print("\n✅ Saved: X_train_bal.npy, Y_train_bal.npy")
print("Balanced Train Shape:", X_train_bal.shape)


Before balancing:
Non-cancer: 6463
Cancer: 1549

After balancing:
Non-cancer: 6463
Cancer: 6463

✅ Saved: X_train_bal.npy, Y_train_bal.npy
Balanced Train Shape: (12926, 19080)


In [13]:
X_train[20]

array([0, 0, 0, ..., 0, 0, 1], shape=(19080,), dtype=object)

In [14]:
print(X_train.shape, Y_train.shape)
print(X_test.shape, Y_test.shape)

(8012, 19080) (8012,)
(2003, 19080) (2003,)


In [16]:
print("Train:", np.bincount(Y_train_bal))
print("Test:", np.bincount(Y_test))

Train: [6463 6463]
Test: [1598  405]
